# Pause Detection Evaluation against Punctuation Ground-Truth (Optimized)
This notebook evaluates the pause-detection accuracy of the backend pipeline against punctuation ground-truth using the FLEURS dataset (en_us, test split).

**Optimization Status:** This notebook is configured to run in an optimized mode. It loads **only the ASR model** and runs pause detection. It skips the WhiStress stress detection and Pitch stylization models entirely, reducing memory by ~2-3 GB and speed by 10x-50x.

**Colab Compatibility:** If run in Google Colab, this notebook will automatically attempt to mount Google Drive for persistent file storage. If Google Drive mounting fails or is skipped, it gracefully falls back to local Colab storage (`/content/fleurs_eval`) so the execution is not interrupted.


In [ ]:
# Run this cell to install required dependencies on cloud GPU environments
!pip install -q pandas numpy matplotlib librosa faster-whisper soundfile torch pydantic huggingface_hub


## 1. Imports and Configuration


In [ ]:
import os
import sys
import json
import tarfile
import string
import difflib
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# --- Google Colab Repository and Drive Setup ---
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab environment detected.")
    
    # Try mounting Google Drive with fallback to Colab local disk
    try:
        print("Mounting Google Drive...")
        from google.colab import drive
        drive.mount('/content/drive')
        FLEURS_DIR = "/content/drive/MyDrive/fleurs_eval"
        print("Successfully mounted Google Drive.")
    except Exception as e:
        print("WARNING: Google Drive mounting failed! Error:", e)
        print("Falling back to local Colab storage (/content/fleurs_eval).")
        print("Note: Data in local storage will not persist if the Colab runtime restarts.")
        FLEURS_DIR = "/content/fleurs_eval"
        
    os.makedirs(FLEURS_DIR, exist_ok=True)
    
    # Clone the repository under the active directory if not present
    if not os.path.exists("./backend") and not os.path.exists("./prosody_interface/backend"):
        print("Cloning repository: https://github.com/Abel-Jacob/prosody_interface.git")
        !git clone https://github.com/Abel-Jacob/prosody_interface.git
        if os.path.exists("prosody_interface"):
            os.chdir("prosody_interface")
            print("Changed directory to:", os.getcwd())
    elif os.path.exists("./prosody_interface/backend") and not os.path.exists("./backend"):
        os.chdir("prosody_interface")
        print("Changed directory to:", os.getcwd())
    else:
        print("Repository already present in directory.")
else:
    # Local path selection
    FLEURS_DIR = r"C:\Users\DELL\Downloads" if os.path.exists(r"C:\Users\DELL\Downloads") else "."

# --- Acquire FLEURS dataset files ---
TEST_TSV_PATH = os.path.join(FLEURS_DIR, "test.tsv")
TEST_TAR_PATH = os.path.join(FLEURS_DIR, "test.tar.gz")

if IN_COLAB:
    # Only download from HF if the files don't already exist in FLEURS_DIR
    if not os.path.exists(TEST_TSV_PATH) or not os.path.exists(TEST_TAR_PATH):
        print("Downloading FLEURS dataset files from Hugging Face...")
        from huggingface_hub import hf_hub_download
        
        try:
            tsv_path = hf_hub_download(
                repo_id="google/fleurs",
                filename="data/en_us/test.tsv",
                repo_type="dataset",
                local_dir=FLEURS_DIR
            )
            tar_path = hf_hub_download(
                repo_id="google/fleurs",
                filename="data/en_us/audio/test.tar.gz",
                repo_type="dataset",
                local_dir=FLEURS_DIR
            )
            TEST_TSV_PATH = tsv_path
            TEST_TAR_PATH = tar_path
            print("Download completed successfully.")
        except Exception as e:
            print("Error downloading from Hugging Face:", e)
    else:
        print("FLEURS dataset files already present in FLEURS_DIR.")

# Target folder to extract FLEURS audio files (under FLEURS_DIR for persistence)
AUDIO_EXTRACT_DIR = os.path.join(FLEURS_DIR, "fleurs_extracted")

# Output files directory (under FLEURS_DIR for persistence)
OUTPUT_DIR = os.path.join(FLEURS_DIR, "pipeline_results")
ERRORS_FILE = os.path.join(FLEURS_DIR, "errors.json")
RESULTS_CSV_PATH = os.path.join(FLEURS_DIR, "results.csv")

# --- Evaluation parameters ---
TOLERANCE_SEC = 0.3  # greedy matching tolerance in seconds
N_SAMPLES = 50        # Number of random files to process (set to None for all)

# --- Backend Path Configuration ---
# Dynamically resolve the backend path relative to the root folder
BACKEND_DIR = os.path.abspath("./backend")
if not os.path.exists(BACKEND_DIR) and os.path.exists("./prosody_interface/backend"):
    BACKEND_DIR = os.path.abspath("./prosody_interface/backend")

print("Using BACKEND_DIR:", BACKEND_DIR)
print("Using FLEURS_DIR:", os.path.abspath(FLEURS_DIR))

# Ensure backend and vendor folders are in sys.path
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

vendor_path = os.path.join(BACKEND_DIR, "vendor")
if vendor_path not in sys.path:
    sys.path.insert(0, vendor_path)


## 2. Load ASR Model and Define Optimized Pipeline Wrapper
We load only the ASR model to save VRAM and execution time. We then implement a minimal `run_pipeline` function that calls ASR and pause-detection code directly.


In [ ]:
# Ensure backend paths are loaded (handles out-of-order execution)
import sys
import os
BACKEND_DIR = os.path.abspath("./backend")
if not os.path.exists(BACKEND_DIR) and os.path.exists("./prosody_interface/backend"):
    BACKEND_DIR = os.path.abspath("./prosody_interface/backend")
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
vendor_path = os.path.join(BACKEND_DIR, "vendor")
if vendor_path not in sys.path:
    sys.path.insert(0, vendor_path)

from pipeline.asr import load_asr_model

print("Loading faster-whisper model...")
asr_model = load_asr_model()
print("ASR model loaded successfully.")

# Import necessary pipeline modules & schemas
from pipeline.asr import transcribe_chunk
from pipeline.merge import group_words_by_punctuation
from pipeline.prosody_pause import PauseAnalyzer
from schemas import WordResult

def run_pipeline(audio_path: str) -> dict:
    # Runs the optimized backend speech pipeline (ASR + Pause analyzer only) on a single audio file.
    import librosa
    
    # Stage 1: Load audio (16kHz mono)
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Stage 2: Transcribe
    asr_result = transcribe_chunk(
        audio=audio,
        model=asr_model,
        language="en",
        is_live=False
    )
    
    # Convert ASR words to flat list of dicts for PauseAnalyzer
    all_word_dicts = [
        {
            "word": w["word"],
            "start": round(w["start"], 3),
            "end": round(w["end"], 3)
        }
        for w in asr_result.get("words", [])
    ]
    
    # Call pause_analyzer ONCE on the full flat list of words
    pause_analyzer = PauseAnalyzer()
    pause_analyzer.setup({})
    res = pause_analyzer.analyze(np.array([]), all_word_dicts)
    
    # Convert to WordResult objects with computed pause & hesitation data
    raw_words = []
    if "word_pauses" in res:
        pause_words = res["word_pauses"]
        for i, w in enumerate(asr_result.get("words", [])):
            pause_after = 0.0
            is_hesitation = False
            if i < len(pause_words):
                pause_after = pause_words[i].get("pause_after", 0.0)
                is_hesitation = pause_words[i].get("is_hesitation", False)
                
            raw_words.append(
                WordResult(
                    word=w["word"],
                    start=round(w["start"], 3),
                    end=round(w["end"], 3),
                    confidence=round(w.get("confidence", 1.0), 3),
                    stressed=False,
                    stress_score=0.0,
                    pause_after=pause_after,
                    is_hesitation=is_hesitation
                )
            )
            
    # Group into grammatical phrases (required for full_transcription and phrase_index assignment)
    # Words will preserve their computed pause_after values
    grammatical_phrases = group_words_by_punctuation(raw_words)
    
    # Generate full transcription
    full_transcription = " ".join([p.text for p in grammatical_phrases])
    
    # Flatten word list
    words_list = []
    word_counter = 0
    for p_idx, phrase in enumerate(grammatical_phrases):
        for w in phrase.words:
            words_list.append({
                "word_index": word_counter,
                "word": w.word,
                "start_time": w.start,
                "end_time": w.end,
                "phrase_index": p_idx,
                "asr_confidence": w.confidence,
                "stressed": w.stressed,
                "stress_score": w.stress_score,
                "pause_after": w.pause_after,
                "is_hesitation": w.is_hesitation
            })
            word_counter += 1
            
    # Format to match exact frontend JSON structure
    json_words = []
    for w in words_list:
        json_words.append({
            "word": w["word"],
            "start_time": w["start_time"],
            "end_time": w["end_time"],
            "stressed": w["stressed"],
            "stress_score_pct": f"{round((w['stress_score'] or 0.0) * 100)}%",
            "word_index": w["word_index"],
            "phrase_index": w["phrase_index"],
            "asr_confidence_pct": f"{round((w['asr_confidence'] or 1.0) * 100)}%",
            "is_hesitation": w["is_hesitation"]
        })
        
        # Insert pause object if pause exceeds 0.5s (frontend logic threshold)
        if w.get("pause_after") and w.get("pause_after") > 0.5:
            json_words.append({
                "pause": f"{w['pause_after']:.2f}s"
            })
            
    return {
        "full_transcription": full_transcription,
        "word_level_timestamps_and_stress": json_words
    }


## 3. Extraction of FLEURS Audio files
We extract audio files from the `test.tar.gz` archive to a local folder `fleurs_extracted`.


In [ ]:
if not os.path.exists(AUDIO_EXTRACT_DIR):
    os.makedirs(AUDIO_EXTRACT_DIR)

audio_files_dir = os.path.join(AUDIO_EXTRACT_DIR, "test")
if os.path.exists(audio_files_dir) and len(os.listdir(audio_files_dir)) > 100:
    print(f"Audio files already extracted in {audio_files_dir}. Skipping extraction.")
else:
    print(f"Extracting {TEST_TAR_PATH} to {AUDIO_EXTRACT_DIR}... (This might take a moment)")
    with tarfile.open(TEST_TAR_PATH, "r:gz") as tar:
        tar.extractall(path=AUDIO_EXTRACT_DIR)
    print("Extraction complete.")


## 4. Batch Processing Loop
We read `test.tsv`, pick a random sample of `N_SAMPLES` files, run the pipeline on them, save JSON results in `pipeline_results`, and track errors in `errors.json`. The process is fully resumable.


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Read test.tsv with columns
columns = ["id", "filename", "raw_transcription", "transcription", "graphemes", "num_samples", "gender"]
df = pd.read_csv(TEST_TSV_PATH, sep="\\t", header=None, names=columns)
print(f"Loaded {len(df)} rows from {TEST_TSV_PATH}")

# Subsample if N_SAMPLES is set
if N_SAMPLES is not None:
    random.seed(42)
    sample_indices = random.sample(range(len(df)), min(N_SAMPLES, len(df)))
    eval_df = df.iloc[sample_indices].copy()
    print(f"Configured to process a random sample of {len(eval_df)} files.")
else:
    eval_df = df.copy()
    print("Configured to process all files.")

# Load existing errors
errors_dict = {}
if os.path.exists(ERRORS_FILE):
    try:
        with open(ERRORS_FILE, "r") as f:
            errors_dict = json.load(f)
    except:
        pass

processed_count = 0
error_count = len(errors_dict)
total_to_process = len(eval_df)

for idx, row in eval_df.iterrows():
    filename = row["filename"]
    json_filename = os.path.splitext(filename)[0] + ".json"
    json_path = os.path.join(OUTPUT_DIR, json_filename)
    
    # Check if already processed
    if os.path.exists(json_path):
        processed_count += 1
        if processed_count % 10 == 0 or processed_count == total_to_process:
            print(f"{processed_count}/{total_to_process} done, {error_count} errors (Skipped existing)")
        continue
        
    audio_path = os.path.join(AUDIO_EXTRACT_DIR, "test", filename)
    
    if not os.path.exists(audio_path):
        error_msg = f"Audio file not found at {audio_path}"
        errors_dict[filename] = error_msg
        error_count += 1
        with open(ERRORS_FILE, "w") as f:
            json.dump(errors_dict, f, indent=2)
        processed_count += 1
        continue
        
    try:
        # Run pipeline
        result = run_pipeline(audio_path)
        
        # Save JSON output
        with open(json_path, "w") as f:
            json.dump(result, f, indent=2)
            
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)}"
        errors_dict[filename] = error_msg
        error_count += 1
        with open(ERRORS_FILE, "w") as f:
            json.dump(errors_dict, f, indent=2)
            
    processed_count += 1
    if processed_count % 10 == 0 or processed_count == total_to_process:
        print(f"{processed_count}/{total_to_process} done, {error_count} errors")


## 5. Extract Pauses and Ground-Truth Points
We extract pauses predicted by our pipeline and align the ground truth transcription with system word timestamps using `difflib.SequenceMatcher` to assign timestamps to punctuation points.


In [ ]:
def extract_system_pauses(data: dict) -> list[dict]:
    # Extracts pauses predicted by the pipeline.
    pauses = []
    word_list = data.get("word_level_timestamps_and_stress", [])
    last_word_idx = None
    last_word_end = 0.0
    
    for entry in word_list:
        if "word" in entry:
            last_word_idx = entry["word_index"]
            last_word_end = entry["end_time"]
        elif "pause" in entry:
            duration_str = entry["pause"]
            try:
                duration = float(duration_str.rstrip("s"))
            except ValueError:
                duration = 0.0
            
            if last_word_idx is not None:
                pauses.append({
                    "after_word_index": last_word_idx,
                    "time": last_word_end,
                    "duration": duration
                })
    return pauses

def extract_groundtruth_pause_points(raw_transcription: str, word_list: list[dict]) -> list[dict]:
    # Extracts punctuation marks in raw_transcription and aligns them to word_list.
    # Filter out pause entries
    sys_words = [w for w in word_list if "word" in w]
    
    # Split raw transcription by whitespace
    raw_words = raw_transcription.split()
    
    target_puncts = {'.', ',', '?', '!', ';', ':'}
    
    raw_clean = []
    raw_punct_info = [] # (clean_word, ending_punct)
    
    for rw in raw_words:
        clean = rw.lower()
        ending_punct = None
        
        # Check if word ends with any target punctuation
        while clean and clean[-1] not in string.ascii_letters and clean[-1] not in string.digits:
            char = clean[-1]
            if char in target_puncts:
                ending_punct = char
                break
            clean = clean[:-1]
            
        clean_aligned = rw.lower().translate(str.maketrans('', '', string.punctuation))
        raw_clean.append(clean_aligned)
        raw_punct_info.append(ending_punct)
        
    sys_clean = [w["word"].lower().translate(str.maketrans('', '', string.punctuation)) for w in sys_words]
    
    # Sequence align raw_clean and sys_clean
    matcher = difflib.SequenceMatcher(None, raw_clean, sys_clean)
    matching_blocks = matcher.get_matching_blocks()
    
    alignment = {}
    for block in matching_blocks:
        for i in range(block.size):
            alignment[block.a + i] = block.b + i
            
    gt_points = []
    for raw_idx, punct in enumerate(raw_punct_info):
        if punct is not None:
            sys_idx = alignment.get(raw_idx)
            if sys_idx is not None:
                sys_word = sys_words[sys_idx]
                gt_points.append({
                    "word_index": sys_word["word_index"],
                    "punct": punct,
                    "time": sys_word["end_time"]
                })
                
    return gt_points


## 6. Evaluation and Score Matching
We implement greedy nearest-time matching within a tolerance window of `TOLERANCE_SEC` to score true positives (TP), false positives (FP), and false negatives (FN).


In [ ]:
def evaluate(gt_points, sys_pauses, tolerance_sec=0.3) -> dict:
    # Performs greedy nearest-time matching within tolerance_sec.
    gt_sorted = sorted(gt_points, key=lambda x: x["time"])
    sys_sorted = sorted(sys_pauses, key=lambda x: x["time"])
    
    matched_sys_indices = set()
    matched_pairs = []
    
    tp = 0
    fn = 0
    
    for gt in gt_sorted:
        gt_time = gt["time"]
        best_sys_idx = None
        min_diff = float("inf")
        
        for idx, sys in enumerate(sys_sorted):
            if idx in matched_sys_indices:
                continue
            
            diff = abs(sys["time"] - gt_time)
            if diff <= tolerance_sec:
                if diff < min_diff:
                    min_diff = diff
                    best_sys_idx = idx
                    
        if best_sys_idx is not None:
            matched_sys_indices.add(best_sys_idx)
            tp += 1
            matched_pairs.append({
                "gt": gt,
                "sys": sys_sorted[best_sys_idx],
                "diff": min_diff
            })
        else:
            fn += 1
            
    fp = len(sys_sorted) - len(matched_sys_indices)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matched_pairs": matched_pairs
    }


## 7. Run Evaluation and Aggregate Results
We run the evaluation for each processed file, micro-average overall results, write them to `results.csv`, compute F1 scores for each punctuation type, and show the visualizations.


In [ ]:
file_results = []

PUNCT_CATEGORIES = {
    '.': 'period',
    ',': 'comma',
    '?': 'question mark',
    '!': 'exclamation (other)',
    ';': 'semicolon (other)',
    ':': 'colon (other)'
}

def get_punct_group(punct):
    group = PUNCT_CATEGORIES.get(punct, 'other')
    if 'other' in group:
        return 'other'
    return group

punct_counts = {
    'period': {'TP': 0, 'FP': 0, 'FN': 0},
    'comma': {'TP': 0, 'FP': 0, 'FN': 0},
    'question mark': {'TP': 0, 'FP': 0, 'FN': 0},
    'other': {'TP': 0, 'FP': 0, 'FN': 0}
}

detected_durations = {
    'period': [],
    'comma': [],
    'question mark': [],
    'other': []
}

for idx, row in eval_df.iterrows():
    filename = row["filename"]
    raw_transcription = row["raw_transcription"]
    
    json_filename = os.path.splitext(filename)[0] + ".json"
    json_path = os.path.join(OUTPUT_DIR, json_filename)
    
    if not os.path.exists(json_path):
        continue
        
    with open(json_path, "r") as f:
        data = json.load(f)
        
    sys_pauses = extract_system_pauses(data)
    gt_points = extract_groundtruth_pause_points(raw_transcription, data.get("word_level_timestamps_and_stress", []))
    
    res = evaluate(gt_points, sys_pauses, tolerance_sec=TOLERANCE_SEC)
    
    file_results.append({
        "filename": filename,
        "TP": res["TP"],
        "FP": res["FP"],
        "FN": res["FN"],
        "precision": res["precision"],
        "recall": res["recall"],
        "f1": res["f1"]
    })
    
    matched_sys_indices = set()
    for pair in res["matched_pairs"]:
        punct = pair["gt"]["punct"]
        group = get_punct_group(punct)
        punct_counts[group]['TP'] += 1
        
        sys_pause_dur = pair["sys"]["duration"]
        detected_durations[group].append(sys_pause_dur)
        
        for s_idx, sys_p in enumerate(sys_pauses):
            if sys_p["time"] == pair["sys"]["time"] and sys_p["duration"] == pair["sys"]["duration"]:
                matched_sys_indices.add(s_idx)
                break
                
    matched_gt_times = {pair["gt"]["time"] for pair in res["matched_pairs"]}
    for gt in gt_points:
        if gt["time"] not in matched_gt_times:
            punct = gt["punct"]
            group = get_punct_group(punct)
            punct_counts[group]['FN'] += 1
            
    for s_idx, sys_p in enumerate(sys_pauses):
        if s_idx in matched_sys_indices:
            continue
        if gt_points:
            closest_gt = min(gt_points, key=lambda x: abs(x["time"] - sys_p["time"]))
            group = get_punct_group(closest_gt["punct"])
        else:
            group = 'other'
        punct_counts[group]['FP'] += 1

# Create Summary DataFrame
df_results = pd.DataFrame(file_results)

# Micro-average overall
total_TP = df_results["TP"].sum()
total_FP = df_results["FP"].sum()
total_FN = df_results["FN"].sum()

overall_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0.0
overall_recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0.0
overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0

df_summary = df_results.copy()
aggregate_row = pd.DataFrame([{
    "filename": "OVERALL (Micro-averaged)",
    "TP": total_TP,
    "FP": total_FP,
    "FN": total_FN,
    "precision": overall_precision,
    "recall": overall_recall,
    "f1": overall_f1
},])
df_summary = pd.concat([df_summary, aggregate_row], ignore_index=True)

# Save to CSV
df_summary.to_csv(RESULTS_CSV_PATH, index=False)

# Breakdown calculations
breakdown_rows = []
for group, counts in punct_counts.items():
    tp = counts['TP']
    fp = counts['FP']
    fn = counts['FN']
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    
    breakdown_rows.append({
        "Punctuation Type": group,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })
df_breakdown = pd.DataFrame(breakdown_rows)


## 8. View Results and Plot Visualizations


In [ ]:
print("=== FILE-LEVEL EVALUATION SUMMARY ===")
display(df_summary.tail(15))

print("\n=== BREAKDOWN BY PUNCTUATION TYPE ===")
display(df_breakdown)

# Bar chart of F1 score by punctuation type
plt.figure(figsize=(8, 5))
plt.bar(df_breakdown["Punctuation Type"], df_breakdown["F1 Score"], color=['#4F46E5', '#06B6D4', '#10B981', '#F59E0B'])
plt.title("Pause Detection F1 Score by Punctuation Type")
plt.xlabel("Punctuation Type")
plt.ylabel("F1 Score")
plt.ylim(0, 1.05)
for idx, val in enumerate(df_breakdown["F1 Score"]):
    plt.text(idx, val + 0.02, f"{val:.3f}", ha='center', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Table of mean detected pause duration by punctuation type
mean_durations = []
for group, durs in detected_durations.items():
    mean_dur = np.mean(durs) if durs else 0.0
    mean_durations.append({
        "Punctuation Type": group,
        "Count of Pauses": len(durs),
        "Mean Detected Pause Duration (sec)": round(mean_dur, 3)
    })
df_durations = pd.DataFrame(mean_durations)

print("\n=== MEAN DETECTED PAUSE DURATION BY PUNCTUATION TYPE ===")
display(df_durations)
